
# Predicción orbital 3D de un satélite: MLP vs RNN vs GRU vs LSTM

Objetivo: comparar cómo distintos modelos aprenden una **secuencia temporal 3D**.

La tarea será:

- observar una ventana pasada de una órbita sintética
- predecir la **posición futura** del satélite
- comparar:
  - **MLP**
  - **RNN**
  - **GRU**
  - **LSTM**


In [1]:
# !pip -q install plotly

import math
import random
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, Markdown

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda



## 1. Generación de órbitas sintéticas

- órbitas elípticas alrededor de un foco
- inclinación orbital
- ligera precesión / rotación del plano
- pequeña oscilación fuera del plano

In [2]:

def rotation_matrix_x(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[1, 0, 0],
                     [0, c, -s],
                     [0, s, c]])

def rotation_matrix_z(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]])

def make_orbit_sequence(
    n_steps=220,
    a=None,          # semieje mayor
    e=None,          # excentricidad
    inc=None,        # inclinación
    omega=None,      # rotación del plano orbital
    phase=None,      # fase inicial
    wobble_amp=None, # oscilación pequeña
    wobble_freq=None,
    drift=None       # ligera precesión
):
    if a is None: a = np.random.uniform(1.0, 1.8)
    if e is None: e = np.random.uniform(0.05, 0.45)
    if inc is None: inc = np.random.uniform(0.15, 1.1)
    if omega is None: omega = np.random.uniform(0.0, 2*np.pi)
    if phase is None: phase = np.random.uniform(0.0, 2*np.pi)
    if wobble_amp is None: wobble_amp = np.random.uniform(0.01, 0.08)
    if wobble_freq is None: wobble_freq = np.random.uniform(1.5, 4.5)
    if drift is None: drift = np.random.uniform(-0.01, 0.01)

    b = a * np.sqrt(1 - e**2)
    t = np.linspace(0, 2*np.pi, n_steps)

    pts = []
    for i, tau in enumerate(t):
        th = tau + phase

        # Elipse base en el plano orbital
        x = a * np.cos(th) - a * e
        y = b * np.sin(th)
        z = wobble_amp * np.sin(wobble_freq * th)

        p = np.array([x, y, z])

        # Rotación del plano + ligera precesión
        rot = rotation_matrix_z(omega + drift * i) @ rotation_matrix_x(inc)
        p = rot @ p

        pts.append(p)

    pts = np.array(pts, dtype=np.float32)
    return pts

def build_dataset(n_orbits=500, n_steps=220):
    return np.stack([make_orbit_sequence(n_steps=n_steps) for _ in range(n_orbits)])

all_orbits = build_dataset(n_orbits=5000, n_steps=220)
all_orbits.shape


(560, 220, 3)

In [3]:

# Visualizamos algunas órbitas sintéticas
fig = go.Figure()
for i in range(6):
    orb = all_orbits[i]
    fig.add_trace(go.Scatter3d(
        x=orb[:,0], y=orb[:,1], z=orb[:,2],
        mode="lines",
        name=f"Órbita {i+1}"
    ))

fig.update_layout(
    title="Órbitas sintéticas 3D",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="data"
    ),
    height=700
)
fig.show()



## 2. Construcción de ventanas temporales

Usaremos:

- `input_len`: número de pasos observados
- `pred_len`: número de pasos futuros a predecir

Todos los modelos resolverán exactamente la **misma tarea**:
predecir una secuencia futura de posiciones 3D.


In [4]:

INPUT_LEN = 30
PRED_LEN = 20

train_orbits = all_orbits[:420]
val_orbits   = all_orbits[420:490]
test_orbits  = all_orbits[490:]

def make_windows(orbits, input_len=30, pred_len=20, stride=3):
    X, Y = [], []
    total_len = input_len + pred_len
    for orb in orbits:
        for start in range(0, len(orb) - total_len + 1, stride):
            x = orb[start:start+input_len]
            y = orb[start+input_len:start+input_len+pred_len]
            X.append(x)
            Y.append(y)
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X_train, Y_train = make_windows(train_orbits, INPUT_LEN, PRED_LEN, stride=3)
X_val, Y_val     = make_windows(val_orbits, INPUT_LEN, PRED_LEN, stride=3)
X_test, Y_test   = make_windows(test_orbits, INPUT_LEN, PRED_LEN, stride=3)

X_train.shape, Y_train.shape


((23940, 30, 3), (23940, 20, 3))

In [5]:

# Estandarización: importante para entrenar bien
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train_2d = X_train.reshape(-1, 3)
Y_train_2d = Y_train.reshape(-1, 3)

scaler_x.fit(X_train_2d)
scaler_y.fit(Y_train_2d)

def scale_X(X):
    return scaler_x.transform(X.reshape(-1, 3)).reshape(X.shape).astype(np.float32)

def scale_Y(Y):
    return scaler_y.transform(Y.reshape(-1, 3)).reshape(Y.shape).astype(np.float32)

def inverse_Y(Ys):
    return scaler_y.inverse_transform(Ys.reshape(-1, 3)).reshape(Ys.shape)

X_train_s = scale_X(X_train)
X_val_s   = scale_X(X_val)
X_test_s  = scale_X(X_test)

Y_train_s = scale_Y(Y_train)
Y_val_s   = scale_Y(Y_val)
Y_test_s  = scale_Y(Y_test)

print(X_train_s.shape, Y_train_s.shape)


(23940, 30, 3) (23940, 20, 3)


In [6]:

class OrbitDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

BATCH_SIZE = 128

train_ds = OrbitDataset(X_train_s, Y_train_s)
val_ds   = OrbitDataset(X_val_s, Y_val_s)
test_ds  = OrbitDataset(X_test_s, Y_test_s)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)



## 3. Modelos

### MLP
El MLP no procesa la secuencia de forma recurrente.  
Simplemente aplana la ventana pasada:

\[
(input\_len, 3) \rightarrow input\_len \times 3
\]

y produce todos los puntos futuros a la vez.

### RNN / GRU / LSTM
Estos sí procesan la secuencia paso a paso y condensan el contexto temporal en su estado interno.


In [7]:

class MLPForecaster(nn.Module):
    def __init__(self, input_len=30, pred_len=20, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_len * 3, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, pred_len * 3)
        )
        self.pred_len = pred_len

    def forward(self, x):
        # x: [B, T, 3]
        out = self.net(x.reshape(x.size(0), -1))
        return out.reshape(x.size(0), self.pred_len, 3)

class RNNForecaster(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=96, pred_len=20):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, batch_first=True, nonlinearity="tanh")
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, pred_len * 3)
        )
        self.pred_len = pred_len

    def forward(self, x):
        out, h = self.rnn(x)
        last = out[:, -1, :]
        y = self.head(last)
        return y.reshape(x.size(0), self.pred_len, 3)

class GRUForecaster(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=96, pred_len=20):
        super().__init__()
        self.rnn = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, pred_len * 3)
        )
        self.pred_len = pred_len

    def forward(self, x):
        out, h = self.rnn(x)
        last = out[:, -1, :]
        y = self.head(last)
        return y.reshape(x.size(0), self.pred_len, 3)

class LSTMForecaster(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=96, pred_len=20):
        super().__init__()
        self.rnn = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, pred_len * 3)
        )
        self.pred_len = pred_len

    def forward(self, x):
        out, (h, c) = self.rnn(x)
        last = out[:, -1, :]
        y = self.head(last)
        return y.reshape(x.size(0), self.pred_len, 3)

models = {
    "MLP": MLPForecaster(INPUT_LEN, PRED_LEN, hidden=256).to(device),
    "RNN": RNNForecaster(3, 96, PRED_LEN).to(device),
    "GRU": GRUForecaster(3, 96, PRED_LEN).to(device),
    "LSTM": LSTMForecaster(3, 96, PRED_LEN).to(device),
}
models


{'MLP': MLPForecaster(
   (net): Sequential(
     (0): Linear(in_features=90, out_features=256, bias=True)
     (1): ReLU()
     (2): Linear(in_features=256, out_features=256, bias=True)
     (3): ReLU()
     (4): Linear(in_features=256, out_features=60, bias=True)
   )
 ),
 'RNN': RNNForecaster(
   (rnn): RNN(3, 96, batch_first=True)
   (head): Sequential(
     (0): Linear(in_features=96, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=60, bias=True)
   )
 ),
 'GRU': GRUForecaster(
   (rnn): GRU(3, 96, batch_first=True)
   (head): Sequential(
     (0): Linear(in_features=96, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=60, bias=True)
   )
 ),
 'LSTM': LSTMForecaster(
   (rnn): LSTM(3, 96, batch_first=True)
   (head): Sequential(
     (0): Linear(in_features=96, out_features=96, bias=True)
     (1): ReLU()
     (2): Linear(in_features=96, out_features=60, bias=True)
   )
 )}


## 4. Entrenamiento
Usaremos pérdida MSE sobre toda la trayectoria futura.


In [8]:

def evaluate_model(model, loader, criterion):
    model.eval()
    losses = []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            losses.append(loss.item())
    return float(np.mean(losses))

def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": []}
    best_state = None
    best_val = float("inf")

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_loss = evaluate_model(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"Epoch {epoch:02d} | train={train_loss:.5f} | val={val_loss:.5f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return history

EPOCHS = 20
histories = {}

for name, model in models.items():
    print("\n" + "="*70)
    print(f"Entrenando {name}")
    histories[name] = train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3)



Entrenando MLP
Epoch 01 | train=0.07105 | val=0.00936
Epoch 02 | train=0.00580 | val=0.00637
Epoch 03 | train=0.00478 | val=0.00958
Epoch 04 | train=0.00467 | val=0.00522
Epoch 05 | train=0.00380 | val=0.00540
Epoch 06 | train=0.00349 | val=0.00480
Epoch 07 | train=0.00321 | val=0.00453
Epoch 08 | train=0.00312 | val=0.00764
Epoch 09 | train=0.00312 | val=0.00486
Epoch 10 | train=0.00278 | val=0.00387
Epoch 11 | train=0.00265 | val=0.00324
Epoch 12 | train=0.00248 | val=0.00845
Epoch 13 | train=0.00275 | val=0.00499
Epoch 14 | train=0.00235 | val=0.00351
Epoch 15 | train=0.00223 | val=0.00482
Epoch 16 | train=0.00229 | val=0.00390
Epoch 17 | train=0.00222 | val=0.00594
Epoch 18 | train=0.00219 | val=0.00306
Epoch 19 | train=0.00198 | val=0.00315
Epoch 20 | train=0.00192 | val=0.00304

Entrenando RNN
Epoch 01 | train=0.15778 | val=0.03251
Epoch 02 | train=0.01727 | val=0.01807
Epoch 03 | train=0.01117 | val=0.01499
Epoch 04 | train=0.00854 | val=0.00872
Epoch 05 | train=0.00727 | val=0

In [9]:

# Curvas de entrenamiento
fig = go.Figure()

for name, hist in histories.items():
    fig.add_trace(go.Scatter(
        y=hist["val_loss"],
        mode="lines+markers",
        name=f"{name} val"
    ))

fig.update_layout(
    title="Curvas de validación",
    xaxis_title="Epoch",
    yaxis_title="MSE",
    height=500
)
fig.show()



## 5. Evaluación en test
Mediremos:
- **MSE**
- **RMSE**
- **MAE**

sobre toda la trayectoria futura.


In [10]:

def predict_numpy(model, Xs):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(Xs), 512):
            xb = torch.tensor(Xs[i:i+512], dtype=torch.float32, device=device)
            pb = model(xb).cpu().numpy()
            preds.append(pb)
    return np.concatenate(preds, axis=0)

rows = []
predictions = {}

for name, model in models.items():
    pred_s = predict_numpy(model, X_test_s)
    pred = inverse_Y(pred_s)
    true = Y_test

    predictions[name] = pred

    mse = mean_squared_error(true.reshape(-1, 3), pred.reshape(-1, 3))
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(true.reshape(-1, 3), pred.reshape(-1, 3))

    rows.append({
        "Modelo": name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae
    })

results_df = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
results_df


,Modelo,MSE,RMSE,MAE
0,LSTM,0.001608,0.040095,0.028152
1,RNN,0.002215,0.047059,0.033846
2,GRU,0.002219,0.047102,0.033027
3,MLP,0.002228,0.047199,0.035075


In [11]:

fig = px.bar(
    results_df,
    x="Modelo",
    y="RMSE",
    title="Comparación de error en test (RMSE)"
)
fig.update_layout(height=500)
fig.show()

results_df


,Modelo,MSE,RMSE,MAE
0,LSTM,0.001608,0.040095,0.028152
1,RNN,0.002215,0.047059,0.033846
2,GRU,0.002219,0.047102,0.033027
3,MLP,0.002228,0.047199,0.035075



## 6. Visualización 3D de una predicción concreta

Vamos a elegir un ejemplo de test y superponer:
- **pasado observado**
- **futuro real**
- **futuro predicho**


In [12]:

sample_idx = 25

past = X_test[sample_idx]
future_true = Y_test[sample_idx]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=past[:,0], y=past[:,1], z=past[:,2],
    mode="lines+markers",
    name="Pasado observado"
))

fig.add_trace(go.Scatter3d(
    x=future_true[:,0], y=future_true[:,1], z=future_true[:,2],
    mode="lines+markers",
    name="Futuro real"
))

for name in ["MLP", "RNN", "GRU", "LSTM"]:
    pred = predictions[name][sample_idx]
    fig.add_trace(go.Scatter3d(
        x=pred[:,0], y=pred[:,1], z=pred[:,2],
        mode="lines",
        name=f"Predicción {name}"
    ))

fig.update_layout(
    title=f"Predicción orbital 3D para una muestra de test (idx={sample_idx})",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="data"
    ),
    height=800
)
fig.show()


In [13]:

# Una vista separada por modelo para comparar mejor
for name in ["MLP", "RNN", "GRU", "LSTM"]:
    pred = predictions[name][sample_idx]

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=past[:,0], y=past[:,1], z=past[:,2],
        mode="lines+markers",
        name="Pasado observado"
    ))
    fig.add_trace(go.Scatter3d(
        x=future_true[:,0], y=future_true[:,1], z=future_true[:,2],
        mode="lines+markers",
        name="Futuro real"
    ))
    fig.add_trace(go.Scatter3d(
        x=pred[:,0], y=pred[:,1], z=pred[:,2],
        mode="lines+markers",
        name=f"Predicción {name}"
    ))
    fig.update_layout(
        title=f"{name} — predicción orbital 3D",
        scene=dict(aspectmode="data"),
        height=700
    )
    fig.show()



## 7. Error a lo largo del horizonte de predicción


In [14]:

def rmse_by_horizon(y_true, y_pred):
    # y_true, y_pred: [N, pred_len, 3]
    errs = []
    for h in range(y_true.shape[1]):
        mse = mean_squared_error(y_true[:, h, :], y_pred[:, h, :])
        errs.append(math.sqrt(mse))
    return np.array(errs)

fig = go.Figure()
for name in ["MLP", "RNN", "GRU", "LSTM"]:
    errs = rmse_by_horizon(Y_test, predictions[name])
    fig.add_trace(go.Scatter(
        x=np.arange(1, PRED_LEN + 1),
        y=errs,
        mode="lines+markers",
        name=name
    ))

fig.update_layout(
    title="RMSE por horizonte de predicción",
    xaxis_title="Paso futuro",
    yaxis_title="RMSE",
    height=500
)
fig.show()


## 8. Inferencia autoregresiva paso a paso

Hasta ahora los modelos predicen los `PRED_LEN` pasos a la vez (**predicción directa multi-step**).  
Ahora vamos a simular un caso más realista: **el modelo predice un paso, lo reinyectamos como entrada, y repetimos**.

Aqui se ve el problema de la **deriva acumulada**:
- un pequeño error en el paso 1 contamina el paso 2
- y ese error vuelve a entrar al modelo
- con el horizonte, la trayectoria se puede desviar bastante

Para hacerlo con los modelos ya entrenados, usaremos solo el **primer punto predicho** en cada iteración.


In [15]:
def predict_autoregressive_numpy(model, Xs, pred_len=PRED_LEN, batch_size=256):
    """
    Rollout autoregresivo usando modelos ya entrenados en modo direct multi-step.
    En cada iteración:
      1) el modelo ve la ventana actual [B, T, 3]
      2) tomamos solo su primera predicción [B, 1, 3]
      3) la añadimos al final de la ventana y descartamos el punto más antiguo
    """
    model.eval()
    preds_all = []

    with torch.no_grad():
        for i in range(0, len(Xs), batch_size):
            xb = torch.tensor(Xs[i:i+batch_size], dtype=torch.float32, device=device)
            window = xb.clone()
            future = []

            for _ in range(pred_len):
                pred_full = model(window)          # [B, PRED_LEN, 3]
                next_step = pred_full[:, 0:1, :]   # nos quedamos con el siguiente paso inmediato
                future.append(next_step.cpu().numpy())
                window = torch.cat([window[:, 1:, :], next_step], dim=1)

            preds_all.append(np.concatenate(future, axis=1))

    return np.concatenate(preds_all, axis=0)

autoregressive_predictions = {}
autoregressive_rows = []

for name, model in models.items():
    pred_ar_s = predict_autoregressive_numpy(model, X_test_s, pred_len=PRED_LEN)
    pred_ar = inverse_Y(pred_ar_s)
    autoregressive_predictions[name] = pred_ar

    mse = mean_squared_error(Y_test.reshape(-1, 3), pred_ar.reshape(-1, 3))
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(Y_test.reshape(-1, 3), pred_ar.reshape(-1, 3))

    autoregressive_rows.append({
        'Modelo': name,
        'Modo': 'Autoregresivo',
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae
    })

autoregressive_df = pd.DataFrame(autoregressive_rows).sort_values('RMSE')
autoregressive_df


,Modelo,Modo,MSE,RMSE,MAE
3,LSTM,Autoregresivo,0.041509,0.203738,0.111010
0,MLP,Autoregresivo,0.058070,0.240977,0.162292
1,RNN,Autoregresivo,0.141253,0.375836,0.230296
2,GRU,Autoregresivo,0.216948,0.465777,0.288295


In [16]:
direct_df = results_df.copy()
direct_df['Modo'] = 'Directo multi-step'
compare_df = pd.concat([direct_df[['Modelo', 'Modo', 'MSE', 'RMSE', 'MAE']], autoregressive_df], axis=0, ignore_index=True)
compare_df


,Modelo,Modo,MSE,RMSE,MAE
0,LSTM,Directo multi-step,0.001608,0.040095,0.028152
1,RNN,Directo multi-step,0.002215,0.047059,0.033846
2,GRU,Directo multi-step,0.002219,0.047102,0.033027
3,MLP,Directo multi-step,0.002228,0.047199,0.035075
4,LSTM,Autoregresivo,0.041509,0.203738,0.111010
5,MLP,Autoregresivo,0.058070,0.240977,0.162292
6,RNN,Autoregresivo,0.141253,0.375836,0.230296
7,GRU,Autoregresivo,0.216948,0.465777,0.288295


In [17]:
fig = px.bar(
    compare_df,
    x='Modelo',
    y='RMSE',
    color='Modo',
    barmode='group',
    title='Error en test: predicción directa vs rollout autoregresivo'
)
fig.update_layout(height=520)
fig.show()


In [18]:
sample_idx = 25
past = X_test[sample_idx]
future_true = Y_test[sample_idx]

for name in ['MLP', 'RNN', 'GRU', 'LSTM']:
    pred_direct = predictions[name][sample_idx]
    pred_ar = autoregressive_predictions[name][sample_idx]

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=past[:,0], y=past[:,1], z=past[:,2], mode='lines+markers', name='Pasado observado'))
    fig.add_trace(go.Scatter3d(x=future_true[:,0], y=future_true[:,1], z=future_true[:,2], mode='lines+markers', name='Futuro real'))
    fig.add_trace(go.Scatter3d(x=pred_direct[:,0], y=pred_direct[:,1], z=pred_direct[:,2], mode='lines+markers', name='Predicción directa'))
    fig.add_trace(go.Scatter3d(x=pred_ar[:,0], y=pred_ar[:,1], z=pred_ar[:,2], mode='lines+markers', name='Predicción autoregresiva'))
    fig.update_layout(
        title=f'{name}: directo vs autoregresivo',
        scene=dict(aspectmode='data'),
        height=720
    )
    fig.show()


In [19]:
fig = go.Figure()
for name in ['MLP', 'RNN', 'GRU', 'LSTM']:
    errs_direct = rmse_by_horizon(Y_test, predictions[name])
    errs_ar = rmse_by_horizon(Y_test, autoregressive_predictions[name])

    fig.add_trace(go.Scatter(
        x=np.arange(1, PRED_LEN + 1), y=errs_direct,
        mode='lines+markers', name=f'{name} directo'
    ))
    fig.add_trace(go.Scatter(
        x=np.arange(1, PRED_LEN + 1), y=errs_ar,
        mode='lines+markers', name=f'{name} autoregresivo',
        line=dict(dash='dash')
    ))

fig.update_layout(
    title='RMSE por horizonte: directo vs autoregresivo',
    xaxis_title='Paso futuro',
    yaxis_title='RMSE',
    height=600
)
fig.show()


## 9. Técnicas para reducir el error autoregresivo

Cuando usamos rollout autoregresivo aparece un **desajuste entre entrenamiento e inferencia**:

- en entrenamiento el modelo suele ver entradas limpias del dataset
- en inferencia ve sus propias predicciones, que contienen error

Posibles maneras de solucionarlo:

### Técnica 1. Predicción directa multi-step
Es justo la que ya tiene este notebook. Evita realimentar errores paso a paso, porque el modelo predice el horizonte completo.

### Técnica 2. Scheduled sampling
Durante el entrenamiento, en vez de alimentar siempre el valor real anterior, a veces alimentas la predicción del propio modelo.

### Técnica 3. Predecir incrementos (`Δx, Δy, Δz`) en vez de posiciones absolutas
Muchas veces es más fácil aprender la dinámica local que la posición absoluta completa.

### Técnica 4. Entrenamiento con ruido en la ventana de entrada
Si el modelo aprende con entradas ligeramente perturbadas, suele ser más robusto cuando sus propias predicciones empiezan a desviarse.

### Técnica 5. Pérdida ponderada por horizonte
Puedes dar más peso a los pasos lejanos si quieres que el modelo cuide mejor el final de la trayectoria.

### Técnica 6. Variables físicas derivadas
Además de `(x, y, z)`, puedes incluir velocidad aproximada, radio orbital o energía aproximada para ayudar a la red a conservar mejor la dinámica.


In [20]:
# Técnica 3: transformar objetivos a incrementos (deltas)
def to_deltas(X, Y):
    """
    Convierte el objetivo futuro absoluto Y en incrementos respecto al último punto observado
    y después entre pasos consecutivos.
    """
    Yd = np.zeros_like(Y)
    Yd[:, 0, :] = Y[:, 0, :] - X[:, -1, :]
    Yd[:, 1:, :] = Y[:, 1:, :] - Y[:, :-1, :]
    return Yd

def deltas_to_positions(last_pos, Yd):
    Y = np.zeros_like(Yd)
    Y[:, 0, :] = last_pos + Yd[:, 0, :]
    for t in range(1, Yd.shape[1]):
        Y[:, t, :] = Y[:, t-1, :] + Yd[:, t, :]
    return Y

# Ejemplo de uso:
Y_train_delta = to_deltas(X_train, Y_train)
Y_val_delta   = to_deltas(X_val, Y_val)
Y_test_delta  = to_deltas(X_test, Y_test)

print('Objetivos delta creados:', Y_train_delta.shape)
print('Luego puedes reentrenar cualquier modelo usando Y_*_delta en vez de Y_*')


Objetivos delta creados: (23940, 20, 3)
Luego puedes reentrenar cualquier modelo usando Y_*_delta en vez de Y_*


In [21]:
# Técnica 4: ruido pequeño en la entrada para ganar robustez
def add_input_noise(X, sigma=0.01):
    noise = np.random.normal(loc=0.0, scale=sigma, size=X.shape).astype(np.float32)
    return X + noise

X_train_s_noisy = add_input_noise(X_train_s, sigma=0.01)
print('Ejemplo de augmentación con ruido listo:', X_train_s_noisy.shape)
print('Puedes entrenar con X_train_s_noisy para mejorar robustez en rollout autoregresivo.')


Ejemplo de augmentación con ruido listo: (23940, 30, 3)
Puedes entrenar con X_train_s_noisy para mejorar robustez en rollout autoregresivo.


In [22]:
# Técnica 5: pérdida ponderada por horizonte
def weighted_horizon_mse(pred, target, gamma=1.5):
    """
    Da más peso a pasos lejanos del horizonte.
    gamma > 1 => los últimos pasos pesan más.
    """
    H = pred.size(1)
    weights = torch.linspace(1.0, gamma, H, device=pred.device).view(1, H, 1)
    return ((pred - target) ** 2 * weights).mean()

# Ejemplo de uso dentro del entrenamiento:
# loss = weighted_horizon_mse(pred, yb, gamma=2.0)

dummy_pred = torch.randn(8, PRED_LEN, 3)
dummy_tgt = torch.randn(8, PRED_LEN, 3)
print('Loss de ejemplo:', float(weighted_horizon_mse(dummy_pred, dummy_tgt, gamma=2.0)))


Loss de ejemplo: 2.9812963008880615


### Esqueleto conceptual de *scheduled sampling*

La idea es simple: en un decodificador paso a paso, durante entrenamiento mezclas dos opciones para el siguiente input:

- con probabilidad $p$, usas el valor real (*teacher forcing*)
- con probabilidad $1-p$, usas la predicción del modelo

Y vas reduciendo $p$ con las épocas.

Pseudo-código:
```python
next_in = y_real[:, t:t+1, :] if random.random() < p_tf else y_pred[:, -1:, :]
```

En este notebook no lo hemos implementado completo porque nuestros modelos actuales predicen todo el horizonte de una vez, no con decodificador recurrente explícito. Pero es una de las mejoras más importantes cuando se trabaja en modo autoregresivo real.
